In [1]:
import pandas as pd
import nltk
import string
import re
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer


In [2]:
# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kengu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kengu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kengu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kengu\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# Display all columns (use the entire DataFrame for display)
df = pd.read_csv('test.txt', sep=';', names=['text', 'label'], engine='python')

Now lets drop unecessary data and use the text and emotions only

In [4]:
# Define emotion label columns
label_cols = [
'joy', 'anger', 'fear', 'sadness', 'love', 'surprise'
]

Lets convert the one-hot encoded emotion columns into single categorical label

In [5]:
emotion_grouping = {
    'joy': 'joy',
    'anger': 'anger',
    'fear': 'fear',
    'sadness': 'sadness',
    'love': 'love',
    'surprise': 'surprise',
    
}

In [6]:
# Convert 'label' column to a list of labels
df['emotion_labels'] = df['label'].apply(lambda x: x.split(';') if pd.notnull(x) else [])

In [7]:
from collections import Counter

# Flatten the emotion_labels lists into a single big list
all_emotion_labels = [label for labels in df['emotion_labels'] for label in labels]

# Count each label
label_counts = Counter(all_emotion_labels)

# Display the results nicely
for label, count in label_counts.items():
    print(f"{label}: {count}")

sadness: 581
joy: 695
fear: 224
anger: 275
love: 159
surprise: 66


In [8]:
# Preview the DataFrame with text and emotion_labels columns
df_preview = df[['text', 'emotion_labels']].head()

df_preview.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]},  # Left-align headers
    {'selector': 'td', 'props': [('text-align', 'left'), ('white-space', 'pre-wrap')]}  # Left-align cells
])

,text,emotion_labels
0,im feeling rather rotten so im not very ambitious right now,['sadness']
1,im updating my blog because i feel shitty,['sadness']
2,i never make her separate from me because i don t ever want her to feel like i m ashamed with her,['sadness']
3,i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived,['joy']
4,i was feeling a little vain when i did this one,['sadness']


In [9]:
# Setup tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


In [10]:
# Step 1: Tokenize (cell by cell)
def tokenize_words(text):
    tokens = word_tokenize(text)
    print(f"Tokens: {tokens}")  # Debugging line
    return tokens

In [11]:
def clean_text(tokens):
    if not tokens:  # Check if the token list is empty
        return ''
    
    cleaned_tokens = []
    for token in tokens:
        token = token.lower()  # lowercase
        token = token.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
        token = re.sub(r'\d+', '', token)  # remove numbers
        # Additional cleaning step for unwanted characters
        token = token.replace("’", "'")  # fix the apostrophe
        token = token.replace("``", "")  # remove any stray backticks
        
        if token:  # Only add non-empty tokens
            cleaned_tokens.append(token)
    
    print(f"Cleaned Tokens: {cleaned_tokens}")  # Debugging print
    return ' '.join(cleaned_tokens)  # Rejoin tokens into a string

In [12]:
# Step 2: Remove stopwords
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words]
    print(f"Removed Stopwords: {filtered}")  # Debugging print
    return ' '.join(filtered)


In [13]:
# Step 3: Lemmatization
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()

def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        return "none"

    # Expand contractions
    expanded_text = contractions.fix(text)

    tokens = word_tokenize(expanded_text.lower())
    normalized = []

    for token in tokens:
        try:
            lemma = lemmatizer.lemmatize(token)
            normalized.append(lemma)
        except:
            continue

    print(f"Normalized Text: {normalized}")
    return ' '.join(normalized) if normalized else 'none'


In [14]:
df['tokens'] = df['text'].apply(tokenize_words)

Tokens: ['im', 'feeling', 'rather', 'rotten', 'so', 'im', 'not', 'very', 'ambitious', 'right', 'now']
Tokens: ['im', 'updating', 'my', 'blog', 'because', 'i', 'feel', 'shitty']
Tokens: ['i', 'never', 'make', 'her', 'separate', 'from', 'me', 'because', 'i', 'don', 't', 'ever', 'want', 'her', 'to', 'feel', 'like', 'i', 'm', 'ashamed', 'with', 'her']
Tokens: ['i', 'left', 'with', 'my', 'bouquet', 'of', 'red', 'and', 'yellow', 'tulips', 'under', 'my', 'arm', 'feeling', 'slightly', 'more', 'optimistic', 'than', 'when', 'i', 'arrived']
Tokens: ['i', 'was', 'feeling', 'a', 'little', 'vain', 'when', 'i', 'did', 'this', 'one']
Tokens: ['i', 'cant', 'walk', 'into', 'a', 'shop', 'anywhere', 'where', 'i', 'do', 'not', 'feel', 'uncomfortable']
Tokens: ['i', 'felt', 'anger', 'when', 'at', 'the', 'end', 'of', 'a', 'telephone', 'call']
Tokens: ['i', 'explain', 'why', 'i', 'clung', 'to', 'a', 'relationship', 'with', 'a', 'boy', 'who', 'was', 'in', 'many', 'ways', 'immature', 'and', 'uncommitted', 'desp

In [15]:
# Apply preprocessing pipeline
df['cleaned_text'] = df['tokens'].apply(clean_text).astype(str)

Cleaned Tokens: ['im', 'feeling', 'rather', 'rotten', 'so', 'im', 'not', 'very', 'ambitious', 'right', 'now']
Cleaned Tokens: ['im', 'updating', 'my', 'blog', 'because', 'i', 'feel', 'shitty']
Cleaned Tokens: ['i', 'never', 'make', 'her', 'separate', 'from', 'me', 'because', 'i', 'don', 't', 'ever', 'want', 'her', 'to', 'feel', 'like', 'i', 'm', 'ashamed', 'with', 'her']
Cleaned Tokens: ['i', 'left', 'with', 'my', 'bouquet', 'of', 'red', 'and', 'yellow', 'tulips', 'under', 'my', 'arm', 'feeling', 'slightly', 'more', 'optimistic', 'than', 'when', 'i', 'arrived']
Cleaned Tokens: ['i', 'was', 'feeling', 'a', 'little', 'vain', 'when', 'i', 'did', 'this', 'one']
Cleaned Tokens: ['i', 'cant', 'walk', 'into', 'a', 'shop', 'anywhere', 'where', 'i', 'do', 'not', 'feel', 'uncomfortable']
Cleaned Tokens: ['i', 'felt', 'anger', 'when', 'at', 'the', 'end', 'of', 'a', 'telephone', 'call']
Cleaned Tokens: ['i', 'explain', 'why', 'i', 'clung', 'to', 'a', 'relationship', 'with', 'a', 'boy', 'who', 'was

In [16]:
df['no_stopwords'] = df['cleaned_text'].apply(remove_stopwords)

Removed Stopwords: ['im', 'feeling', 'rather', 'rotten', 'im', 'ambitious', 'right']
Removed Stopwords: ['im', 'updating', 'blog', 'feel', 'shitty']
Removed Stopwords: ['never', 'make', 'separate', 'ever', 'want', 'feel', 'like', 'ashamed']
Removed Stopwords: ['left', 'bouquet', 'red', 'yellow', 'tulips', 'arm', 'feeling', 'slightly', 'optimistic', 'arrived']
Removed Stopwords: ['feeling', 'little', 'vain', 'one']
Removed Stopwords: ['cant', 'walk', 'shop', 'anywhere', 'feel', 'uncomfortable']
Removed Stopwords: ['felt', 'anger', 'end', 'telephone', 'call']
Removed Stopwords: ['explain', 'clung', 'relationship', 'boy', 'many', 'ways', 'immature', 'uncommitted', 'despite', 'excitement', 'feeling', 'getting', 'accepted', 'masters', 'program', 'university', 'virginia']
Removed Stopwords: ['like', 'breathless', 'feeling', 'reader', 'eager', 'see', 'happen', 'next']
Removed Stopwords: ['jest', 'feel', 'grumpy', 'tired', 'pre', 'menstrual', 'probably', 'week', 'im', 'fit', 'walrus', 'vacatio

In [17]:
df['normalized'] = df['no_stopwords'].apply(normalize_text)

Normalized Text: ['i', 'am', 'feeling', 'rather', 'rotten', 'i', 'am', 'ambitious', 'right']
Normalized Text: ['i', 'am', 'updating', 'blog', 'feel', 'shitty']
Normalized Text: ['never', 'make', 'separate', 'ever', 'want', 'feel', 'like', 'ashamed']
Normalized Text: ['left', 'bouquet', 'red', 'yellow', 'tulip', 'arm', 'feeling', 'slightly', 'optimistic', 'arrived']
Normalized Text: ['feeling', 'little', 'vain', 'one']
Normalized Text: ['can', 'not', 'walk', 'shop', 'anywhere', 'feel', 'uncomfortable']
Normalized Text: ['felt', 'anger', 'end', 'telephone', 'call']
Normalized Text: ['explain', 'clung', 'relationship', 'boy', 'many', 'way', 'immature', 'uncommitted', 'despite', 'excitement', 'feeling', 'getting', 'accepted', 'master', 'program', 'university', 'virginia']
Normalized Text: ['like', 'breathless', 'feeling', 'reader', 'eager', 'see', 'happen', 'next']
Normalized Text: ['jest', 'feel', 'grumpy', 'tired', 'pre', 'menstrual', 'probably', 'week', 'i', 'am', 'fit', 'walrus', 'vaca

In [18]:
# Step 4: Extract bigrams manually from the normalized text
def get_bigrams(text):
    words = text.split()
    bigrams = [(words[i], words[i + 1]) for i in range(len(words) - 1)]
    return bigrams

df['bigrams'] = df['normalized'].apply(get_bigrams)

# View a sample
print(df[['normalized', 'bigrams']].head())


                                          normalized  \
0    i am feeling rather rotten i am ambitious right   
1                     i am updating blog feel shitty   
2    never make separate ever want feel like ashamed   
3  left bouquet red yellow tulip arm feeling slig...   
4                            feeling little vain one   

                                             bigrams  
0  [(i, am), (am, feeling), (feeling, rather), (r...  
1  [(i, am), (am, updating), (updating, blog), (b...  
2  [(never, make), (make, separate), (separate, e...  
3  [(left, bouquet), (bouquet, red), (red, yellow...  
4   [(feeling, little), (little, vain), (vain, one)]  


In [19]:
df['bigrams_text'] = df['bigrams'].apply(lambda x: ' '.join(['_'.join(tup) for tup in x]))
df['combined_text'] = df['normalized'] + ' ' + df['bigrams_text']

In [20]:
# Rename columns for better display
df_display = df[['text','tokens', 'cleaned_text', 'no_stopwords', 'normalized', 'combined_text']].rename(columns={
    'text': 'Original',
    'tokens': 'Tokens',
    'cleaned_text': 'Cleaned',
    'no_stopwords': 'No Stopwords',
    'normalized': 'Normalized',
    'combined_text': 'Combined',
})

# Apply styles: left-align everything (headers and content), and wrap text
styled_df = df_display.head().style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]},  # Header alignment
    {'selector': 'td', 'props': [('text-align', 'left'), ('white-space', 'pre-wrap'), ('font-size', '12px')]}  # Cell alignment
])

styled_df


,Original,Tokens,Cleaned,No Stopwords,Normalized,Combined
0,im feeling rather rotten so im not very ambitious right now,"['im', 'feeling', 'rather', 'rotten', 'so', 'im', 'not', 'very', 'ambitious', 'right', 'now']",im feeling rather rotten so im not very ambitious right now,im feeling rather rotten im ambitious right,i am feeling rather rotten i am ambitious right,i am feeling rather rotten i am ambitious right i_am am_feeling feeling_rather rather_rotten rotten_i i_am am_ambitious ambitious_right
1,im updating my blog because i feel shitty,"['im', 'updating', 'my', 'blog', 'because', 'i', 'feel', 'shitty']",im updating my blog because i feel shitty,im updating blog feel shitty,i am updating blog feel shitty,i am updating blog feel shitty i_am am_updating updating_blog blog_feel feel_shitty
2,i never make her separate from me because i don t ever want her to feel like i m ashamed with her,"['i', 'never', 'make', 'her', 'separate', 'from', 'me', 'because', 'i', 'don', 't', 'ever', 'want', 'her', 'to', 'feel', 'like', 'i', 'm', 'ashamed', 'with', 'her']",i never make her separate from me because i don t ever want her to feel like i m ashamed with her,never make separate ever want feel like ashamed,never make separate ever want feel like ashamed,never make separate ever want feel like ashamed never_make make_separate separate_ever ever_want want_feel feel_like like_ashamed
3,i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived,"['i', 'left', 'with', 'my', 'bouquet', 'of', 'red', 'and', 'yellow', 'tulips', 'under', 'my', 'arm', 'feeling', 'slightly', 'more', 'optimistic', 'than', 'when', 'i', 'arrived']",i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived,left bouquet red yellow tulips arm feeling slightly optimistic arrived,left bouquet red yellow tulip arm feeling slightly optimistic arrived,left bouquet red yellow tulip arm feeling slightly optimistic arrived left_bouquet bouquet_red red_yellow yellow_tulip tulip_arm arm_feeling feeling_slightly slightly_optimistic optimistic_arrived
4,i was feeling a little vain when i did this one,"['i', 'was', 'feeling', 'a', 'little', 'vain', 'when', 'i', 'did', 'this', 'one']",i was feeling a little vain when i did this one,feeling little vain one,feeling little vain one,feeling little vain one feeling_little little_vain vain_one


Now we train the model.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

First we need to convert the text into numerical data for the modle to read

In [22]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1, 3),        # Include unigrams, bigrams, trigrams
    sublinear_tf=True,         # Use logarithmic scaling
    min_df=5,                  # Remove rare words
    max_df=0.9,                # Remove overly common words
)# You can adjust max_features based on your dataset


In [23]:
# Fit and transform the 'normalized' text
X = vectorizer.fit_transform(df['combined_text'])

In [24]:
# Extract labels (using 'emotion_labels' instead of 'broad_labels')
y = df['emotion_labels']  # Now using 'emotion_labels'

In [25]:
print(np.unique(y, return_counts=True))

(array([list(['anger']), list(['fear']), list(['joy']), list(['love']),
       list(['sadness']), list(['surprise'])], dtype=object), array([275, 224, 695, 159, 581,  66], dtype=int64))


Train-Test Split

In [26]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Load the training and testing datasets
train_df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'labels'])
test_df = pd.read_csv('test.txt', sep=';', header=None, names=['text', 'labels'])

# Process the labels (split by ';' for multi-label)
train_df['emotion_labels'] = train_df['labels'].apply(lambda x: x.split(';'))
test_df['emotion_labels'] = test_df['labels'].apply(lambda x: x.split(';'))

# Create the feature set (X) and labels (y)
X_train = train_df['text']
y_train = train_df['emotion_labels']
X_test = test_df['text']
y_test = test_df['emotion_labels']

# Binarize the multi-label target variable using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y_train_bin = mlb.fit_transform(y_train)
y_test_bin = mlb.transform(y_test)

# Now you have binarized labels, ready for model training
print(f"Training features: {X_train.shape}, Test features: {X_test.shape}")
print(f"Training labels: {y_train_bin.shape}, Test labels: {y_test_bin.shape}")



Training features: (16000,), Test features: (2000,)
Training labels: (16000, 6), Test labels: (2000, 6)


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize the text data
vectorizer = TfidfVectorizer(max_features=10000)  # You can adjust the max_features as needed
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Initialize base classifier
base_clf = LogisticRegression(
    solver='liblinear',   # Good for small to medium datasets; try 'saga' for larger ones
    max_iter=1000,        # Increase if convergence warnings appear
    class_weight='balanced'  # Optional: balances weights inversely proportional to class frequencies
)

# Wrap with OneVsRest strategy for multilabel
ovr_clf = OneVsRestClassifier(base_clf)

# Train the model
ovr_clf.fit(X_train_vec, y_train_bin)

# Make predictions (for example, on the test set)
y_pred_bin = ovr_clf.predict(X_test_vec)

# You can print the predictions to see the output
print(y_pred_bin)


[[0 0 0 0 1 0]
 [0 0 0 0 1 0]
 [0 0 0 0 1 0]
 ...
 [0 0 1 0 0 0]
 [0 0 1 0 0 0]
 [0 1 0 0 0 1]]


In [28]:
y_pred_bin = ovr_clf.predict(X_test_vec)

In [29]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions on the test set (binary labels)
y_pred_bin = ovr_clf.predict(X_test_vec)

# Report for multilabel classification (using predicted binary labels)
print(classification_report(y_test_bin, y_pred_bin, target_names=mlb.classes_))

# Hamming score: average per-label accuracy
print(f"Hamming accuracy: {accuracy_score(y_test_bin, y_pred_bin):.3f}")


              precision    recall  f1-score   support

       anger       0.82      0.94      0.88       275
        fear       0.79      0.93      0.86       224
         joy       0.88      0.94      0.91       695
        love       0.65      0.92      0.76       159
     sadness       0.91      0.93      0.92       581
    surprise       0.52      0.86      0.65        66

   micro avg       0.83      0.93      0.88      2000
   macro avg       0.76      0.92      0.83      2000
weighted avg       0.84      0.93      0.88      2000
 samples avg       0.86      0.93      0.88      2000

Hamming accuracy: 0.790


c:\Users\kengu\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
